In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_quality_audit import need_strip

In [2]:
df = pd.read_csv(
    filepath_or_buffer="../data/raw/orders.csv"
)
df.head()

,order_id,customer_id,order_date,ship_date,delivery_date,customer_name,customer_segment,customer_country,customer_city,customer_region,...,tax_amount,total_order_value,profit,shipping_method,delivery_status,payment_method,payment_status,sales_channel,customer_acquisition_channel,campaign
0,ORD1000000,CUST101392,2025-10-28,2025-10-28,2025-10-28,Thomas Moore,Consumer,United States,Los Angeles,North America,...,4.96,98.95,24.87,Same-Day,On Time,DEBIT CARD,Paid,Mobile App,Organic Search,NaN
1,ORD1000001,CUST117918,2025-04-22,2025-04-23,2025-04-26,Cassandra Hays,Consumer,United States,Houston,North America,...,10.96,172.91,52.96,Standard,On Time,Credit Card,Pending,Mobile App,Organic Search,NaN
2,ORD1000002,CUST111893,2025-08-01,2025-08-02,NaN,Steven Torres,Corporate,United States,Houston,North America,...,16.72,255.64,71.17,Standard,In Transit,PayPal,Paid,Marketplace,Social Media,NaN
3,ORD1000003,CUST110447,2025-11-24,2025-11-26,2025-11-28,Michelle Franco,Consumer,United States,Houston,North America,...,6.76,109.43,46.55,Standard,Delayed,Gift Card,Paid,Website,Email Marketing,BLACKFRIDAY25
4,ORD1000004,CUST102862,2025-01-15,2025-01-16,2025-01-18,Taylor Brown,Small Business,United States,New York,North America,...,3.55,60.04,14.76,Standard,On Time,Debit Card,Paid,Website,Email Marketing,LOYALTY_REWARDS


In [3]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 50125
Number of columns: 27


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50125 entries, 0 to 50124
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   order_id                      50125 non-null  str    
 1   customer_id                   50125 non-null  str    
 2   order_date                    50125 non-null  str    
 3   ship_date                     41577 non-null  str    
 4   delivery_date                 36691 non-null  str    
 5   customer_name                 50125 non-null  str    
 6   customer_segment              50125 non-null  str    
 7   customer_country              50125 non-null  str    
 8   customer_city                 49123 non-null  str    
 9   customer_region               49373 non-null  str    
 10  order_status                  50125 non-null  str    
 11  total_items                   50125 non-null  int64  
 12  unique_products               50125 non-null  int64  
 13  product_cate

**Hanlding data types:**

`order_date`, `ship_date`, and `delivery_date` columns are dates but their data type is string in the dataset now and should be fixed.

In [5]:
pd.DataFrame(
    data=np.array([
        df.isnull().sum(),
        (df.isnull().sum() / df.shape[0] * 100).astype(int),
        df.nunique()
	]).T,
    columns=["missing_count", "missing_percent", "unique_values"],
    index=df.columns
).reset_index()

,index,missing_count,missing_percent,unique_values
0,order_id,0,0,50000
1,customer_id,0,0,22129
2,order_date,0,0,1496
3,ship_date,8548,17,976
4,delivery_date,13434,26,982
5,customer_name,0,0,20289
6,customer_segment,0,0,3
7,customer_country,0,0,6
8,customer_city,1002,1,16
9,customer_region,752,1,3


In [6]:
(df['customer_name'].str.strip() != df['customer_name']).sum()

np.int64(1004)

In [7]:
df['customer_segment'].value_counts()

customer_segment
Consumer          30464
Corporate         12046
Small Business     7615
Name: count, dtype: int64

In [8]:
df['customer_country'].value_counts()

customer_country
United States     26687
United Kingdom     7555
Canada             7442
Germany            5145
Australia          2473
USA                 823
Name: count, dtype: int64

In [9]:
df['customer_city'].value_counts().reset_index()

,customer_city,count
0,Houston,6774
1,New York,6755
2,Chicago,6742
3,Los Angeles,6666
4,Birmingham,2511
5,Manchester,2459
6,Toronto,2456
7,London,2436
8,Montreal,2419
9,Vancouver,2414


In [10]:
df['customer_region'].value_counts()

customer_region
North America    34412
Europe           12510
Oceania           2451
Name: count, dtype: int64

In [11]:
df['order_status'].value_counts()

order_status
Delivered     32593
Shipped        4886
Cancelled      4508
Returned       4098
Processing     4040
Name: count, dtype: int64

In [12]:
df['total_items'].value_counts()

total_items
3     13473
2     11537
4     10023
5      5903
1      5114
6      2607
7      1008
8       342
9        97
10       18
11        2
12        1
Name: count, dtype: int64

In [13]:
df['unique_products'].value_counts()

unique_products
2     18265
3     13350
4      7587
1      6138
5      3241
6      1129
7       314
8        77
9        23
10        1
Name: count, dtype: int64

In [14]:
df['subtotal'].describe()

count    50125.000000
mean       106.781126
std         92.508781
min          4.360000
25%         47.110000
50%         80.980000
75%        136.090000
max       1612.350000
Name: subtotal, dtype: float64

In [15]:
df['discount_amount'].describe()

count    49121.000000
mean         8.363563
std         15.797112
min          0.000000
25%          0.000000
50%          0.000000
75%         11.130000
max        277.740000
Name: discount_amount, dtype: float64

In [16]:
df_null_discount = df[[
    'subtotal', 'discount_amount', 'shipping_cost', 'tax_amount', 'total_order_value'
]][df['discount_amount'].isnull()]
df_null_discount['validate_total'] = df['subtotal'] + df['shipping_cost'] + df['tax_amount']
df_null_discount

,subtotal,discount_amount,shipping_cost,tax_amount,total_order_value,validate_total
201,35.02,NaN,14.89,6.65,56.56,56.56
258,220.26,NaN,0.00,22.39,194.64,242.65
261,58.53,NaN,13.30,3.65,69.04,75.48
331,58.54,NaN,4.31,11.71,74.56,74.56
391,131.72,NaN,7.56,9.22,148.50,148.50
...,...,...,...,...,...,...
49993,257.28,NaN,7.55,18.01,282.84,282.84
50010,160.73,NaN,0.00,27.03,162.20,187.76
50067,144.73,NaN,0.00,10.13,154.86,154.86
50097,14.77,NaN,12.84,0.79,24.97,28.40


In [17]:
print(f"Number of columns that their discount amount recorded as null \nbut the real discount amount is zero: {
    (df_null_discount['total_order_value'] == df_null_discount['validate_total']).sum()
}")

Number of columns that their discount amount recorded as null 
but the real discount amount is zero: 381


In [18]:
df['shipping_cost'].describe()

count    49523.000000
mean         7.598589
std          5.950065
min        -26.580000
25%          4.370000
50%          6.390000
75%         12.040000
max         27.000000
Name: shipping_cost, dtype: float64

In [19]:
# indicies of the columns that the shipping cost recorded as negative values
nsc_indicies = df[df['shipping_cost'] < 0].iloc[:, 13:].index

print(f"Number of samples with negative shipping cost :{df.loc[nsc_indicies].shape[0]}")

Number of samples with negative shipping cost :123


In [20]:
def compute_total_order_value_with_negative_shipping_cost(df):
    new_total_order = df['subtotal'] - df['shipping_cost'] + df['tax_amount'] - df['discount_amount']
    return new_total_order

new_total_order_value = compute_total_order_value_with_negative_shipping_cost(df.loc[nsc_indicies])
print(f"Number of samples that the shipping cost recorded as negative wrongly: {
    (new_total_order_value == df.loc[nsc_indicies, 'total_order_value']).sum()
}")

Number of samples that the shipping cost recorded as negative wrongly: 73


In [21]:
samples_to_fix_negative_shipping = df.loc[nsc_indicies][(new_total_order_value == df.loc[nsc_indicies, 'total_order_value']).values].index
print("Below samples can fix by negating the shipping cost")
samples_to_fix_negative_shipping

Below samples can fix by negating the shipping cost


Index([  136,   613,  1651,  4653,  4943,  6013,  6307,  6802,  6805,  7225,
        7963,  8319,  8448,  8902, 11298, 11957, 12537, 13530, 13916, 14929,
       15493, 16900, 17200, 17367, 19425, 19926, 20136, 22948, 23228, 24029,
       25276, 26337, 27008, 27481, 27591, 28935, 29207, 29352, 30076, 30416,
       31063, 31703, 32365, 32914, 33378, 35035, 35418, 35419, 35635, 37209,
       37313, 37967, 38418, 38829, 39238, 39547, 40374, 41083, 41477, 41599,
       41928, 42557, 42989, 43150, 43467, 44307, 45759, 46912, 47015, 47151,
       47692, 48377, 50112],
      dtype='int64')

In [22]:
df.loc[nsc_indicies, [
    'subtotal', 'discount_amount', 'shipping_cost', 'tax_amount', 'total_order_value'
]][(new_total_order_value != df.loc[nsc_indicies, 'total_order_value']).values]

,subtotal,discount_amount,shipping_cost,tax_amount,total_order_value
890,49.52,NaN,-3.91,9.90,63.33
1184,20.41,0.00,-7.90,1.43,29.74
1309,40.53,0.00,-6.12,2.84,49.49
2137,23.91,1.28,-3.74,1.58,27.95
3486,47.46,2.40,-13.40,3.15,61.61
6098,64.91,0.00,-15.87,4.54,85.32
6224,34.06,0.00,-4.74,2.38,41.18
6769,225.08,37.08,-6.82,13.16,207.98
8351,164.84,45.84,-6.90,8.33,134.23
10345,48.82,0.00,-16.83,9.76,75.41


In [23]:
def compute_total(subtotal, discount, shipping, tax):
    return subtotal + shipping + tax - discount

subtotal = 40.53
discount = 0
shipping = 6.12
tax = 2.84

compute_total(subtotal, discount, shipping, tax)

49.489999999999995

In [43]:
df.loc[
    df['shipping_cost'].isnull(),
    [
        'order_date', 'ship_date', 'delivery_date',
        'order_status', 'shipping_cost', 'shipping_method',
        'delivery_status', 'discount_amount'
	]
].isnull().sum()

order_date           0
ship_date           92
delivery_date      140
order_status         0
shipping_cost      602
shipping_method      0
delivery_status      6
discount_amount     12
dtype: int64

In [94]:
df_discount_shipping_null = df[df['discount_amount'].isnull()][df[df['discount_amount'].isnull()]['shipping_cost'].isnull()]
df_discount_shipping_null.loc[:, 'subtotal':]

,subtotal,discount_amount,shipping_cost,tax_amount,total_order_value,profit,shipping_method,delivery_status,payment_method,payment_status,sales_channel,customer_acquisition_channel,campaign
2479,47.95,NaN,NaN,3.36,57.44,21.63,Standard,On Time,Gift Card,Paid,Mobile App,Organic Search,NaN
3312,118.69,NaN,NaN,6.76,103.28,25.65,Express,On Time,PayPal,Paid,Website,Referral,NaN
11074,229.77,NaN,NaN,37.92,243.51,67.78,Standard,On Time,PayPal,Paid,Website,Email Marketing,SUMMER_SALE24
11716,31.41,NaN,NaN,4.24,32.52,1.63,Standard,On Time,Debit Card,Paid,Website,Paid Ads,NaN
11914,78.46,NaN,NaN,8.61,86.86,10.37,Express,On Time,Credit Card,Pending,Mobile App,Social Media,BLACKFRIDAY24
17063,156.13,NaN,NaN,10.93,167.06,66.21,Express,On Time,Gift Card,Paid,Website,Email Marketing,NaN
25613,22.84,NaN,NaN,1.45,26.17,4.82,Economy,On Time,Bank Transfer,Paid,Social Media,Paid Ads,NaN
28250,14.24,NaN,NaN,1.85,21.40,5.82,Standard,On Time,Credit Card,Paid,Website,Paid Ads,NaN
34779,100.08,NaN,NaN,16.91,101.46,33.68,Standard,Not Shipped,Bank Transfer,Failed,Website,Paid Ads,WINTER_CLEAROUT
36564,101.39,NaN,NaN,13.18,129.79,30.35,Express,On Time,Debit Card,Paid,Social Media,Direct,NaN


In [50]:
df.loc[
    df['shipping_cost'].isnull(),
    [
        'order_date', 'ship_date', 'delivery_date',
        'order_status', 'shipping_cost', 'shipping_method',
        'delivery_status', 'discount_amount'
	]
]['delivery_status'].value_counts()

delivery_status
On Time        329
Not Shipped     92
Delayed         75
Returned        52
In Transit      48
Name: count, dtype: int64

In [55]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_date                          0
ship_date                        8548
delivery_date                   13434
customer_name                       0
customer_segment                    0
customer_country                    0
customer_city                    1002
customer_region                   752
order_status                        0
total_items                         0
unique_products                     0
product_categories                  0
subtotal                            0
discount_amount                  1004
shipping_cost                     602
tax_amount                          0
total_order_value                   0
profit                              0
shipping_method                     0
delivery_status                   501
payment_method                    401
payment_status                      0
sales_channel                       0
customer_acquisition_channel        0
campaign    

In [67]:
df.loc[df['ship_date'].isnull(), 'order_status'].value_counts()

order_status
Cancelled     4508
Processing    4040
Name: count, dtype: int64

In [69]:
df['order_status'].value_counts()

order_status
Delivered     32593
Shipped        4886
Cancelled      4508
Returned       4098
Processing     4040
Name: count, dtype: int64

In [70]:
df.loc[df['ship_date'].isnull(), 'delivery_status'].value_counts()

delivery_status
Not Shipped    8469
Name: count, dtype: int64

In [71]:
df['delivery_status'].value_counts()

delivery_status
On Time        27334
Not Shipped     8469
Delayed         4931
In Transit      4827
Returned        4063
Name: count, dtype: int64

In [86]:
df.loc[
    df['delivery_date'].isnull(),
    [
        'ship_date', 'delivery_date',
		'order_status', 'shipping_method',
		'delivery_status'
	]
]['order_status'].value_counts()

order_status
Shipped       4886
Cancelled     4508
Processing    4040
Name: count, dtype: int64

In [87]:
df['order_status'].value_counts()

order_status
Delivered     32593
Shipped        4886
Cancelled      4508
Returned       4098
Processing     4040
Name: count, dtype: int64

In [88]:
df.loc[
    df['delivery_date'].isnull(),
    [
        'ship_date', 'delivery_date',
		'order_status', 'shipping_method',
		'delivery_status'
	]
]['delivery_status'].value_counts()

delivery_status
Not Shipped    8469
In Transit     4827
Name: count, dtype: int64

In [89]:
df['delivery_status'].value_counts()

delivery_status
On Time        27334
Not Shipped     8469
Delayed         4931
In Transit      4827
Returned        4063
Name: count, dtype: int64

In [92]:
df['shipping_method'].value_counts()

shipping_method
Standard    27489
Express     12563
Economy      7576
Same-Day     2497
Name: count, dtype: int64

In [104]:
df['delivery_status'].value_counts()

delivery_status
On Time        27334
Not Shipped     8469
Delayed         4931
In Transit      4827
Returned        4063
Name: count, dtype: int64

In [109]:
df[df['delivery_status'].isnull()].isnull().sum()

order_id                          0
customer_id                       0
order_date                        0
ship_date                        79
delivery_date                   138
customer_name                     0
customer_segment                  0
customer_country                  0
customer_city                     8
customer_region                   9
order_status                      0
total_items                       0
unique_products                   0
product_categories                0
subtotal                          0
discount_amount                   8
shipping_cost                     6
tax_amount                        0
total_order_value                 0
profit                            0
shipping_method                   0
delivery_status                 501
payment_method                    5
payment_status                    0
sales_channel                     0
customer_acquisition_channel      0
campaign                        420
dtype: int64

In [124]:
df['order_status'].value_counts()

order_status
Delivered     32593
Shipped        4886
Cancelled      4508
Returned       4098
Processing     4040
Name: count, dtype: int64

In [130]:
df['payment_method'].value_counts()

payment_method
Credit Card      21528
PayPal           11809
Debit Card        8603
Gift Card         3413
Bank Transfer     2387
credit card        443
CREDIT CARD        413
paypal             268
PAYPAL             242
DEBIT CARD         195
debit card         173
GIFT CARD           82
bank transfer       61
gift card           57
BANK TRANSFER       50
Name: count, dtype: int64

In [152]:
df['payment_status'].value_counts()

payment_status
Paid        38033
Refunded     7225
Pending      3486
Failed       1381
Name: count, dtype: int64

In [154]:
df['sales_channel'].value_counts()

sales_channel
Website         24947
Mobile App      15117
Marketplace      7537
Social Media     2524
Name: count, dtype: int64

In [157]:
df['customer_acquisition_channel'].value_counts()

customer_acquisition_channel
Organic Search     13945
Paid Ads           11191
Social Media        7105
Referral            6914
Email Marketing     5961
Direct              5009
Name: count, dtype: int64

In [159]:
df['campaign'].value_counts()

campaign
SPRING_LAUNCH25    1511
LOYALTY_REWARDS    1484
BLACKFRIDAY25      1479
SUMMER_SALE24      1401
BLACKFRIDAY24      1352
WINTER_CLEAROUT    1326
Name: count, dtype: int64

In [167]:
# exact duplicates
df['order_id'].duplicated().sum()

np.int64(125)

**data quality audit:**

- At first there are 125 exact duplicates that we will drop them.

- There are 125 exact duplicates based-on the business key for this dataset that is `order_id`. order_id should be unique.

- `ship_date` & `delivery_date` includes many missing values near one fourth of the data and should be fixed and the reason of their missing should be investigated.

- `ship_date` column has 8,548 missing values, we can see that all the rows that have no ship_date have no delivery_date too, So 8,548 missing values from 13,434 missing values of delivery_date column are because of this and they are not at random.

- By analyzing we conclude that samples without `ship_date` is because the orders_status of the order is **cancelled** or **processing** and it's obvious that their delivery status is **not shipped**.

- So we conclude that `delivery_status` missing values are ship_date missing values plus orders that their order_status is **shipped** and their delivery_status is **in transit**.

- In `customer_name` there are 1,004 samples of the names of the customers that included white spaces and needs to be fixed. As a best practice we make all the names lower case cause it's easier to work with.

- In `customer_segment` there is no problem just make all the categories lower case.

- `customer_country` has a quality issue, we have united states two times and there is a semantic problem that needs to be fixed USA and United States are the same. Make all the country names lower case and fix this problem too.

- In `customer_city` there are 1,002 missing values and needs to be fixed. Make all the categories lower case. We can try to impute the missing columns using informations from `customer_country` column.

- In `customer_region` column there are 752 missing values that we can impute them using the customer_country column. Make all the categories lower case.

- `order_status` has no problem just make all the categories lower case.

- `total_items` & `unique_products` columns has no problem.

- `product_categories` is a column that has text data about the category of the products that exist in the order. Our analysis for now is not product related and we are analyzing the whole orders and each row represent a single order so this column is not needed and we can drop it.

- `discount_amount` column has 1,004 missing values and we derived that 381 of them are zero discount but we have a better way to impute the real value for these missing values based-on the below formula:
$$
	\text{total order value} = \text{subtotal} + \text{shipping cost}
	+ \text{tax amount} - \text{discount amount}
$$
so we can use this formula and impute all the missing values of the discount_amount column.

- By analyzing the `shipping_cost` we find out that there are 123 negative shipping cost values that collected wrongly and they should be positive and it's just an insertion problem.

- In `shipping_cost` columns there are 602 missing values. We know that in 12 both discount_amount and shipping_cost are missing so we put them away now. First we impute the discount_amount rows and then use the same formula to impute the shipping_cost, this 12 remaining rows are another problem. We can use machine learning methods to estimate the shipping_cost and then fill the remaining discount_amount. Indicies of those 12 samples are: 

[2479,  3312, 11074, 11716, 11914, 17063, 25613, 28250, 34779, 36564,
45197, 45825]

- `shipping_method` column has no problem just make all categories lower case.

- `delivery_status` column has 501 missing values after imputing all other missing values I will use a classification algorithm to estimate the missing values for these 501 missing values.

- `payment_method` column has quality issues in the categories of payment method we should make all the categories lower case and the case inconsistancy will be handled in this way. after that we can see that this column has 401 missing values that we will use a **classification algorithm** try use the best values to fill the missing values and do the best estimation or we can use **KNNImputer** too.

- `payment_status` colum has no problem and we just make all the categories lower case.

- `sales_channel` & `customer_acquisition_channel` columns have no problem and just normalize the texts.

- About 82% of the campaign column is missing and it makes sense cause campaign runs for a short period of time and sales in this short periods has the campaing column so we can keep it just for the analysis of campaign for later if needed.

In [214]:
# data validations

assert (df['total_items'] < 0).sum() == 0, "Invalid total_items Found"
assert (df['unique_products'] < 0).sum() == 0, "Invalid unique_products Found"
assert (df['subtotal'] < 0).sum() == 0, "Invalid subtotal Found"
assert (df['discount_amount'] < 0).sum() == 0, "Invalid discount_amount Found (negative)"
assert (df['discount_amount'] > df['total_order_value']).sum() == 0, "Invalid discount_amount Found (bigger than total order value)"
assert (df['tax_amount'] < 0).sum() == 0, "Invalid tax_amount Found"
assert (df['total_order_value'] < 0).sum() == 0, "Invalid total_order_value Found"
assert (df['profit'] > df['total_order_value']).sum() == 0, "Invalid profit Found"
assert (df['shipping_cost'] < 0).sum() == 0, "Invalid shipping_cost Found"

AssertionError: Invalid shipping_cost Found

Negative shipping cost values found that need to be fixed.